In [1]:
import torch
import torch.nn as nn
from torch.nn.functional import relu
from image_processing import read_img
from sklearn.model_selection import train_test_split
import os
import cv2
import numpy as np
from math import ceil

device="cuda" if torch.cuda.is_available() else "cpu"
torch.manual_seed(42)
print(f"Used device: {device}")

#CONSTANTS
INPUT_DIR:str="./images"
MANUAL_DIR:str="./manual1"
MASK_DIR:str="./mask"

#PATHS
input_images=[INPUT_DIR+"/"+file.name for file in os.scandir(INPUT_DIR) if file.is_file()]
manual_images=[MANUAL_DIR+"/"+file.name for file in os.scandir(MANUAL_DIR) if file.is_file()]
mask_images=[MASK_DIR+"/"+file.name for file in os.scandir("./mask") if file.is_file()]

Used device: cuda


In [2]:
class UNet(nn.Module):
    def __init__(self, n_class):
        super().__init__()

        # input: 572x572x3
        self.e11 = nn.Conv2d(3, 64, kernel_size=3, padding=1) # output: 570x570x64
        self.e12 = nn.Conv2d(64, 64, kernel_size=3, padding=1) # output: 568x568x64
        self.pool1 = nn.MaxPool2d(kernel_size=2, stride=2) # output: 284x284x64

        # input: 284x284x64
        self.e21 = nn.Conv2d(64, 128, kernel_size=3, padding=1) # output: 282x282x128
        self.e22 = nn.Conv2d(128, 128, kernel_size=3, padding=1) # output: 280x280x128
        self.pool2 = nn.MaxPool2d(kernel_size=2, stride=2) # output: 140x140x128

        # input: 140x140x128
        self.e31 = nn.Conv2d(128, 256, kernel_size=3, padding=1) # output: 138x138x256
        self.e32 = nn.Conv2d(256, 256, kernel_size=3, padding=1) # output: 136x136x256
        self.pool3 = nn.MaxPool2d(kernel_size=2, stride=2) # output: 68x68x256

        # input: 68x68x256
        self.e41 = nn.Conv2d(256, 512, kernel_size=3, padding=1) # output: 66x66x512
        self.e42 = nn.Conv2d(512, 512, kernel_size=3, padding=1) # output: 64x64x512
        self.pool4 = nn.MaxPool2d(kernel_size=2, stride=2) # output: 32x32x512

        # input: 32x32x512
        self.e51 = nn.Conv2d(512, 1024, kernel_size=3, padding=1) # output: 30x30x1024
        self.e52 = nn.Conv2d(1024, 1024, kernel_size=3, padding=1) # output: 28x28x1024


        # Decoder
        self.upconv1 = nn.ConvTranspose2d(1024, 512, kernel_size=2, stride=2)
        self.d11 = nn.Conv2d(1024, 512, kernel_size=3, padding=1)
        self.d12 = nn.Conv2d(512, 512, kernel_size=3, padding=1)

        self.upconv2 = nn.ConvTranspose2d(512, 256, kernel_size=2, stride=2)
        self.d21 = nn.Conv2d(512, 256, kernel_size=3, padding=1)
        self.d22 = nn.Conv2d(256, 256, kernel_size=3, padding=1)

        self.upconv3 = nn.ConvTranspose2d(256, 128, kernel_size=2, stride=2)
        self.d31 = nn.Conv2d(256, 128, kernel_size=3, padding=1)
        self.d32 = nn.Conv2d(128, 128, kernel_size=3, padding=1)

        self.upconv4 = nn.ConvTranspose2d(128, 64, kernel_size=2, stride=2)
        self.d41 = nn.Conv2d(128, 64, kernel_size=3, padding=1)
        self.d42 = nn.Conv2d(64, 64, kernel_size=3, padding=1)

        # Output layer
        self.outconv = nn.Conv2d(64, n_class, kernel_size=1)

    def forward(self, x):
        # Encoder
        xe11 = relu(self.e11(x))
        xe12 = relu(self.e12(xe11))
        xp1 = self.pool1(xe12)

        xe21 = relu(self.e21(xp1))
        xe22 = relu(self.e22(xe21))
        xp2 = self.pool2(xe22)

        xe31 = relu(self.e31(xp2))
        xe32 = relu(self.e32(xe31))
        xp3 = self.pool3(xe32)

        xe41 = relu(self.e41(xp3))
        xe42 = relu(self.e42(xe41))
        xp4 = self.pool4(xe42)

        xe51 = relu(self.e51(xp4))
        xe52 = relu(self.e52(xe51))
        
        # Decoder
        xu1 = self.upconv1(xe52)
        xu11 = torch.cat([xu1, xe42], dim=1)
        xd11 = relu(self.d11(xu11))
        xd12 = relu(self.d12(xd11))

        xu2 = self.upconv2(xd12)
        xu22 = torch.cat([xu2, xe32], dim=1)
        xd21 = relu(self.d21(xu22))
        xd22 = relu(self.d22(xd21))

        xu3 = self.upconv3(xd22)
        xu33 = torch.cat([xu3, xe22], dim=1)
        xd31 = relu(self.d31(xu33))
        xd32 = relu(self.d32(xd31))

        xu4 = self.upconv4(xd32)
        xu44 = torch.cat([xu4, xe12], dim=1)
        xd41 = relu(self.d41(xu44))
        xd42 = relu(self.d42(xd41))

        # Output layer
        out = self.outconv(xd42)

        return out

In [3]:
def load_training_patches(image_paths, mask_paths, patch_size=256,percent_of_images:float=0.5):
    x_patches = []
    y_patches = []
    max_id=ceil(len(image_paths)*percent_of_images)
    for idx,(img_p, mask_p) in enumerate(zip(image_paths, mask_paths)):
        img = cv2.imread(str(img_p))
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        mask = cv2.imread(str(mask_p), cv2.IMREAD_GRAYSCALE)

        h, w = img.shape[:2]

        for y in range(0, h - patch_size + 1, patch_size):
            for x in range(0, w - patch_size + 1, patch_size):
                patch_img = img[y : y + patch_size, x : x + patch_size]
                patch_mask = mask[y : y + patch_size, x : x + patch_size]

                if np.max(patch_img) > 15:
                    patch_img = patch_img.astype(np.float32) / 255.0
                    patch_img = np.transpose(patch_img, (2, 0, 1))
                    patch_mask = patch_mask.astype(np.float32) / 255.0
                    patch_mask = np.expand_dims(patch_mask, axis=0)

                    x_patches.append(patch_img)
                    y_patches.append(patch_mask)
        if idx>=max_id:
            break

    return torch.tensor(np.array(x_patches),device=device), torch.tensor(np.array(y_patches),device=device)

In [4]:

print(torch.cuda.get_device_name(0))

total_vram = torch.cuda.get_device_properties(0).total_memory

print(f"VRAM: {total_vram / 1024**3:.2f} GB")

NVIDIA GeForce RTX 4050 Laptop GPU
VRAM: 6.00 GB


In [ ]:
model = UNet(1).to(device)
print(f"The model will be trained on {next(model.parameters()).device}")

epochs=5_000
batch_size=8
criterion = nn.BCEWithLogitsLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)
X,y=load_training_patches(input_images,mask_images,percent_of_images=0.35)
X_train_patches, X_test, Y_train_patches, y_test = train_test_split(X,y, test_size=0.25, random_state=42)
num_train_samples = X_train_patches.size()[0]
old_ang_loss=float("inf")
print("Started the training ")

for epoch in range(epochs):
    model.train()
    epoch_loss = 0

    permutation = torch.randperm(num_train_samples)

    for i in range(0, num_train_samples, batch_size):
        indices = permutation[i : i + batch_size]

        batch_x = X_train_patches[indices].float().to(device)
        batch_y = Y_train_patches[indices].float().to(device)

        pred_y = model(batch_x)

        loss = criterion(pred_y, batch_y)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        epoch_loss += loss.item()

    avg_loss = epoch_loss / (num_train_samples / batch_size)
    if avg_loss<old_ang_loss:
        torch.save(model.state_dict(), 'model_weights_best_found.pth')
    old_ang_loss=avg_loss
    print(f"Epoch: {epoch + 1} z {epochs} Mean error: {avg_loss:.4f}")

In [7]:
torch.save(model.state_dict(), 'model_weights.pth')
print(model.state_dict())

OrderedDict({'e11.weight': tensor([[[[ 0.1518,  0.1643, -0.0406],
          [ 0.1814, -0.0376,  0.0433],
          [-0.0891,  0.1176,  0.1741]],

         [[-0.1371,  0.1713,  0.0400],
          [ 0.1463,  0.0301,  0.0967],
          [-0.0231,  0.1523,  0.0323]],

         [[-0.0855,  0.0534, -0.0845],
          [-0.0182, -0.0739,  0.1318],
          [-0.1476, -0.0845, -0.0502]]],


        [[[-0.1116,  0.0222, -0.1860],
          [ 0.1779, -0.1594,  0.1526],
          [ 0.0361, -0.0584,  0.1230]],

         [[ 0.0335,  0.1589,  0.0245],
          [-0.0572,  0.0552, -0.0487],
          [ 0.0845,  0.1753,  0.1147]],

         [[-0.0808,  0.1144,  0.0378],
          [ 0.1011, -0.1140, -0.1871],
          [-0.0710, -0.1442,  0.1613]]],


        [[[ 0.0599,  0.0842,  0.0654],
          [ 0.0012,  0.1551, -0.1322],
          [ 0.0166, -0.1268,  0.0639]],

         [[-0.0619,  0.0634, -0.0356],
          [ 0.1640, -0.1097, -0.1103],
          [-0.1104,  0.1775,  0.0686]],

         [[ 0.189